## Induced matching on Gaussian Mixture models

In this notebook, we explore subsamples of a sample taken by a Gaussian Mixture model, and consider their matching diagrams. In particular, we look at extreme cases when the matching distances are either minimum or maximum. 

To start, we load the necessary packages.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

import scipy.spatial.distance as dist
import itertools

import tdqual.topological_data_quality_0 as tdqual

import os 
plots_dir = os.path.join("plots", "gaussian-mixture")
os.makedirs(plots_dir, exist_ok=True)

Next, we take a sample from a Gaussian Mixture model. For this, we take about $200$ points and consider $5$ clusters which are well separated, three of which have a high probability while the other two are almost negligible. Nonetheless, we check that the sample contains at least one element on each cluster.

In [ ]:
from sklearn.mixture import GaussianMixture

# 1. Define the parameters
n_samples = 700
n_components = 8

# Define centers to ensure they are clearly separated
# (x, y) coordinates for the 5 clusters
centers = np.array([
    [0, 0], [10, 10], [0, 10], [10, 0], [5, 5], [15,5], [20,10], [20,0]
])


weights = np.array([0.24, 0.14, 0.24, 0.005, 0.005, 0.24, 0.005, 0.12])
while(True):
    # 2. Initialize the GMM
    gmm = GaussianMixture(n_components=n_components, random_state=42)
    
    # Manually set the parameters to force the desired distribution
    gmm.means_ = centers
    gmm.weights_ = weights
    # Set small covariances so clusters remain tight and separated
    gmm.covariances_ = np.array([np.eye(2) * 0.5 for _ in range(n_components)])
    # Required for initialization
    gmm.precisions_cholesky_ = np.linalg.cholesky(np.linalg.inv(gmm.covariances_))
    
    # 3. Sample from the model
    Z, y = gmm.sample(n_samples)
    if len(np.unique(y))>6:
        break

Next, we visualise our sample.

In [ ]:
fig, ax = plt.subplots(ncols=1, figsize=(3,3))
ax.scatter(Z[:,0], Z[:,1], color=mpl.colormaps["RdBu"](1/1.3), s=60, marker="o", zorder=2)
ax.set_axis_off()
plt.savefig(os.path.join(plots_dir,"points_0.png"))

Then, we take a few subsamples of a fixed size.

In [ ]:
size_sample_X = 30
rng = np.random.default_rng(seed=4)
num_subsamples = 300
list_X_Z = []
for i in range(num_subsamples):
    # Take sample of points and reorder
    X_indices = rng.choice(Z.shape[0],size_sample_X, replace=False)
    X_indices_compl = list(set(range(Z.shape[0]))-set(X_indices))
    # Reorder Z and get X
    X =Z[X_indices]
    Z_new = np.vstack((X, Z[X_indices_compl]))
    list_X_Z.append((X, Z_new))




For illustration purposes, we depict the first subsample.

In [ ]:
fig, ax = plt.subplots(ncols=1, figsize=(4,4))
X, Z_pair = list_X_Z[0]
ax.scatter(X[:,0], X[:,1], color=mpl.colormaps["RdBu"](0.3/1.3), s=60, marker="o", zorder=3, label="X")
ax.scatter(Z_pair[:,0], Z_pair[:,1], color=mpl.colormaps["RdBu"](1/1.3), s=40, marker="X", zorder=2, label="Z")
ax.set_title("Sample and subsample")
plt.legend(loc="upper right")
plt.savefig(os.path.join(plots_dir, "samples_0.png"))

Next, we compute the matching diagrams for all subsamples.

In [ ]:
from tdqual.matching_bootstrap import process_pairs_L1


finite_L1_weights, inf_L1_weights = process_pairs_L1(list_X_Z)

In [ ]:
print(f"finite weights, min:{np.min(finite_L1_weights):2.4f}, max: {np.max(finite_L1_weights):2.4f}")
print(f"   inf weights, min:{np.min(inf_L1_weights):2.4f}, max: {np.max(inf_L1_weights):2.4f}")

In [ ]:
fig, ax = plt.subplots(ncols=2,nrows=2, figsize=(8,7))

data = [finite_L1_weights, inf_L1_weights]
for i in range(2):
    # Plot min
    idx = np.argmin(data[i])
    X, Z_pair = list_X_Z[idx]
    ax[0,i].scatter(X[:,0], X[:,1], color=mpl.colormaps["RdBu"](0.3/1.3), s=60, marker="o", zorder=3)
    ax[0,i].scatter(Z_pair[:,0], Z_pair[:,1], color=mpl.colormaps["RdBu"](1/1.3), s=40, marker="X", zorder=2)
    ax[0,i].set_title(["fin ", "inf "][i] + "min: " + f"{data[i][idx]:2.4f}")
    # Plot max
    idx = np.argmax(data[i])
    X, Z_pair = list_X_Z[idx]
    ax[1,i].scatter(X[:,0], X[:,1], color=mpl.colormaps["RdBu"](0.3/1.3), s=60, marker="o", zorder=3)
    ax[1,i].scatter(Z_pair[:,0], Z_pair[:,1], color=mpl.colormaps["RdBu"](1/1.3), s=40, marker="X", zorder=2)
    ax[1,i].set_title(["fin ", "inf "][i] + "max: " + f"{data[i][idx]:2.4f}")

plt.savefig(os.path.join(plots_dir, "maxs_and_mins.png"))

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(8,3))

# Plot median fin
indices_ordenados = np.argsort(data[0])
n = len(data[0])
idx = indices_ordenados[n//2]
X, Z_pair = list_X_Z[idx]
ax[0].scatter(X[:,0], X[:,1], color=mpl.colormaps["RdBu"](0.3/1.3), s=60, marker="o", zorder=3)
ax[0].scatter(Z_pair[:,0], Z_pair[:,1], color=mpl.colormaps["RdBu"](1/1.3), s=40, marker="X", zorder=2)
ax[0].set_title("fin median:" + f"{data[0][idx]:2.4f}")
# Plot median inf
indices_ordenados = np.argsort(data[1])
n = len(data[1])
idx = indices_ordenados[n//2]
X, Z_pair = list_X_Z[idx]
ax[1].scatter(X[:,0], X[:,1], color=mpl.colormaps["RdBu"](0.3/1.3), s=60, marker="o", zorder=3)
ax[1].scatter(Z_pair[:,0], Z_pair[:,1], color=mpl.colormaps["RdBu"](1/1.3), s=40, marker="X", zorder=2)
ax[1].set_title("inf median:" + f"{data[1][idx]:2.4f}")
plt.savefig(os.path.join(plots_dir, "medians.png"))

## Bootstrap

We are not going to use bootstrap since the results are very similar to the ones without it.

Next, we will rank the samples using bootstrap.

In [ ]:
def bootstrap_subsamples_Z(Z, size_subsample=Z.shape[0], nb_times=80):
    sub = []
    # construct a list of nb_times x nb_points
    for sub_idx in range(nb_times):
        Z_idx = np.random.choice(range(Z.shape[0]), Z.shape[0])
        Z_sub = Z[Z_idx]
        # Append subsample 
        sub.append(Z_sub)
    # range over all subsamples
    return sub

In [ ]:
means_fin = []
means_inf = []
for i in range(len(list_X_Z[:10])):
    print(i)
    X, Z = list_X_Z[i]
    sub = bootstrap_subsamples_Z(Z, size_subsample=X.shape[0], nb_times=80)
    sub_pairs_X = [(X, np.vstack((X,Z_sub))) for Z_sub in sub]
    # Matching from X to (X U Z)
    finite_L1_weights, inf_L1_weights = process_pairs_L1(sub_pairs_X)
    mean_fin_X = np.mean(finite_L1_weights)
    mean_inf_X = np.mean(inf_L1_weights)
    # Matching from Z to (X U Z)
    sub_pairs_Z = [(Z_sub, np.vstack((Z_sub,X))) for Z_sub in sub]
    finite_L1_weights, inf_L1_weights = process_pairs_L1(sub_pairs_Z)
    mean_fin_Z = np.mean(finite_L1_weights)
    mean_inf_Z = np.mean(inf_L1_weights)
    # Save means of means
    means_fin.append(np.mean([mean_fin_X, mean_fin_Z]))
    means_inf.append(np.mean([mean_inf_X, mean_inf_Z]))

In [ ]:
fig, ax = plt.subplots(ncols=2,nrows=2, figsize=(8,7))

data = [means_fin, means_inf]
for i in range(2):
    # Plot min
    idx = np.argmin(data[i])
    X, Z = list_X_Z[idx]
    ax[0,i].scatter(X[:,0], X[:,1], color=mpl.colormaps["RdBu"](0.3/1.3), s=60, marker="o", zorder=3)
    ax[0,i].scatter(Z[:,0], Z[:,1], color=mpl.colormaps["RdBu"](1/1.3), s=40, marker="X", zorder=2)
    ax[0,i].set_title(["fin ", "inf "][i] + "min: " + f"{data[i][idx]:2.4f}")
    # Plot max
    idx = np.argmax(data[i])
    X, Z = list_X_Z[idx]
    ax[1,i].scatter(X[:,0], X[:,1], color=mpl.colormaps["RdBu"](0.3/1.3), s=60, marker="o", zorder=3)
    ax[1,i].scatter(Z[:,0], Z[:,1], color=mpl.colormaps["RdBu"](1/1.3), s=40, marker="X", zorder=2)
    ax[1,i].set_title(["fin ", "inf "][i] + "max: " + f"{data[i][idx]:2.4f}")

plt.savefig(os.path.join(plots_dir, "maxs_and_mins.png"))